# 🎬 Studio Video GPU (free, on Colab)

Turns this Colab into a **real AI video server** for your Studio app — free T4 GPU.

## How to use
1. **Runtime → Change runtime type → T4 GPU** (important!)
2. **Runtime → Run all** (or run each cell top to bottom)
3. Wait for the model to download (~2–3 min the first time)
4. The last cell prints a **public URL** like `https://xxxx.trycloudflare.com`
5. Paste that URL into the Studio app → **API key** → *GPU server URL* → Save
6. Video tab → **Real AI (free)** → Generate 🎉

⚠️ **Keep this tab open.** Colab stops the GPU when the tab closes or after ~90 min idle.
Re-run the notebook to get a new URL when that happens.
### If a cell fails with `ImportError`
pip upgraded packages that were already loaded. Do **Runtime → Restart session**,
then run cells 1–5 again (you can skip cell 2 after the first successful install).


In [ ]:
#@title 1️⃣ Check the GPU is on
import subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout)
import torch
assert torch.cuda.is_available(), '❌ No GPU! Runtime → Change runtime type → T4 GPU, then Run all again.'
print('✅ GPU ready:', torch.cuda.get_device_name(0))

In [ ]:
#@title 2️⃣ Install the libraries (~1-2 min)
# Unpinned + upgraded: Colab ships Python 3.12, and an old pinned diffusers
# breaks against the newer huggingface_hub/torch already installed there.
!pip -q install -U diffusers transformers accelerate safetensors huggingface_hub \
    imageio imageio-ffmpeg fastapi 'uvicorn[standard]' nest_asyncio 2>&1 | tail -2
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
import diffusers, huggingface_hub, torch
print('✅ Installed | diffusers', diffusers.__version__,
      '| hub', huggingface_hub.__version__, '| torch', torch.__version__)
print('⚠️  If pip upgraded packages, do: Runtime → Restart session, then run cells 1-5 again.')


In [ ]:
#@title 3️⃣ Load the video model (AnimateDiff-Lightning — fast on T4)
import torch
from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

DEVICE, DTYPE = 'cuda', torch.float16
BASE = 'emilianJR/epiCRealism'          # photoreal SD1.5 base
REPO = 'ByteDance/AnimateDiff-Lightning'
CKPT = 'animatediff_lightning_4step_diffusers.safetensors'

adapter = MotionAdapter().to(DEVICE, DTYPE)
adapter.load_state_dict(load_file(hf_hub_download(REPO, CKPT), device=DEVICE))
pipe = AnimateDiffPipeline.from_pretrained(BASE, motion_adapter=adapter, torch_dtype=DTYPE).to(DEVICE)
pipe.scheduler = EulerDiscreteScheduler.from_config(
    pipe.scheduler.config, timestep_spacing='trailing', beta_schedule='linear')
pipe.enable_vae_slicing()
print('✅ Model loaded — ready to generate video')


In [ ]:
#@title 4️⃣ Start the video server
import io, tempfile, threading, nest_asyncio, uvicorn, imageio, numpy as np
from fastapi import FastAPI
from fastapi.responses import Response, JSONResponse
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

@app.get('/health')
def health():
    return {'ok': True, 'model': 'animatediff-lightning'}

def make_video(prompt: str, steps: int = 4, guidance: float = 1.0) -> bytes:
    out = pipe(prompt=prompt, guidance_scale=guidance, num_inference_steps=steps)
    frames = [np.array(f) for f in out.frames[0]]
    path = tempfile.mktemp(suffix='.mp4')
    imageio.mimsave(path, frames, fps=8, codec='libx264',
                    output_params=['-pix_fmt', 'yuv420p'])
    return open(path, 'rb').read()

@app.get('/generate')
def generate(prompt: str = 'a cinematic scene'):
    """GET /generate?prompt=... -> mp4 bytes (what the Studio app calls)."""
    try:
        return Response(content=make_video(prompt), media_type='video/mp4')
    except Exception as e:
        return JSONResponse({'error': str(e)}, status_code=500)

nest_asyncio.apply()
threading.Thread(
    target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning'),
    daemon=True).start()
print('✅ Server running on :8000')

In [ ]:
#@title 5️⃣ Get your public URL  👉 paste this into the Studio app
import subprocess, re, time, threading

url_box = {}
def run_tunnel():
    p = subprocess.Popen(['cloudflared','tunnel','--url','http://localhost:8000','--no-autoupdate'],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
        if m and 'url' not in url_box:
            url_box['url'] = m.group(0)

threading.Thread(target=run_tunnel, daemon=True).start()
for _ in range(60):
    if 'url' in url_box:
        break
    time.sleep(1)

if 'url' in url_box:
    print('\n' + '='*60)
    print('🎉  YOUR GPU SERVER URL — paste this into the Studio app:')
    print('\n     ' + url_box['url'] + '\n')
    print('    App → API key → "GPU server URL" → Save')
    print('='*60)
    print('\n⚠️  Keep this tab open. The URL dies when Colab stops.')
else:
    print('❌ Tunnel did not start — just re-run this cell.')

In [ ]:
#@title 6️⃣ (Optional) Test it here
from IPython.display import Video, display
vid = make_video('a red sports car driving on a coastal road, cinematic')
open('test.mp4','wb').write(vid)
display(Video('test.mp4', embed=True, width=480))